# CAEd 2026 | Escola de Cidadania Airam Veras
## Laboratório de Estatística Aplicada, Contraprova e Planejamento por Habilidade

**Notebook executável no Google Colab — Engenharia guiada por especificação (SDD) + desenvolvimento guiado por testes (TDD).**

**Disciplinas:** Língua Portuguesa, 1º e 2º anos. **Meta operacional:** pelo menos 80% de estudantes classificados como *Adequado* na aplicação monitorada. **Fontes:** planilha consolidada `CAEd_Airam_Veras_2026_Integrado_DADOS_BRUTO.xlsx` (abas `AVALIACOES_ID`, `RESPOSTAS_ID`, `PARES_ID`, `PAINEL`, `DRIVE_AUDITORIA`).

> **Privacidade:** esta versão inclui uma cópia **pseudonimizada** dos registros analíticos (IDs estáveis, sem nomes), para funcionar sem uploads. IDs estáveis ainda podem ser dados pessoais se alguém possuir a chave de correspondência: não compartilhe resultados individuais publicamente. Os códigos H01...H13 não indicam seus descritores pedagógicos, que devem ser extraídos da matriz oficial antes de recomendar sequências de ensino por conteúdo.

### Modo de uso
1. No Colab, selecione **Ambiente de execução → Executar tudo**: a base pseudonimizada é carregada automaticamente.
2. Para usar uma versão atualizada do Excel, altere `FONTE = "upload"` ou `FONTE = "drive"` na seção de configuração. A estrutura precisa conter as abas exigidas.
3. Confirme que todos os testes aparecem como **OK**. Analise os gráficos, a fila de habilidades e os arquivos exportados.
4. Use as simulações como **cenários condicionais**, não como predições clínicas, pessoais ou de causalidade.

## Etapa 1 — SDD: especificação verificável *antes* da implementação

| Requisito | Contrato/critério de aceitação | Teste |
|---|---|---|
| SDD-01 | Dados contêm abas e colunas esperadas; sem IDs nulos | `test_colunas_obrigatorias` |
| SDD-02 | Identidade do estudante no par: `(Ano, ID)`; nunca comparar registros não pareados | `test_pares_sao_subconjunto`, `test_tamanho_pares` |
| SDD-03 | Acertos e itens por habilidade fecham com avaliação individual | `test_agregacao_acertos` |
| SDD-04 | Erros = itens − acertos; valores válidos | `test_integridade_respostas` |
| SDD-05 | Meta de 80% converte-se em `ceil(0,8 * n)` estudantes | `test_meta_arredondamento`, `test_meta_observada` |
| SDD-06 | Teste pareado com dados adequados; Pearson, Spearman e McNemar exato | `test_mcnemar`, `test_pearson_identidade`, `test_pares_completos` |
| SDD-07 | Priorizar habilidades segundo erros dos estudantes com menor distância até o corte | `test_priorizacao_alvo` |
| SDD-08 | Cenário é limitado pelos itens errados realmente disponíveis; probabilidades ∈ [0,1] | `test_cenario_limites`, `test_poisson_binomial` |
| SDD-09 | Fontes duplicadas ou sem avaliação não aumentam `n` | `test_nao_duplicacao_fonte_drive` |
| SDD-10 | Sinalizar acertos zerados; não inferir ausência, deficiência ou causa | `test_casos_zerados` |

### Premissas e limites obrigatórios

- As etiquetas **referência** e **contraprova** são hipóteses de organização das exportações; não substituem a identificação do ciclo/versão do instrumento.
- O corte **17/22** (1º ano) e **16/22** (2º ano) é **empírico**, observado na contraprova, e não um novo critério oficial do CAEd.
- As aplicações têm diferentes composições de itens/habilidades: correlação mede associação relativa, não crescimento causal nem mudança em escala equacionada.
- Há somente duas aplicações e amostras pequenas. **Não há modelo de previsão longitudinal validado**; reportam-se limites determinísticos e análise de sensibilidade condicionada a probabilidades hipotéticas.
- O arquivo do Drive rotulado Matemática foi auditado com **zero estudantes avaliados e indicação de Língua Portuguesa**. Excluído de análises de Matemática.

In [ ]:
# AMBIENTE: bibliotecas pré-instaladas no Colab; instalação apenas se necessário.
import sys, subprocess, importlib.util
for biblioteca, pacote in [("numpy","numpy"),("pandas","pandas"),("scipy","scipy"),("matplotlib","matplotlib"),("openpyxl","openpyxl")]:
    if importlib.util.find_spec(biblioteca) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pacote])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, zlib, base64, math, io, hashlib, zipfile, shutil, unittest
from pathlib import Path
from scipy.stats import pearsonr, spearmanr, binomtest, binom
from IPython.display import display, Markdown
print("Ambiente de análise pronto. Python:",sys.version.split()[0])

## Etapa 2 — TDD: especificar os testes *antes* do código analítico

A célula a seguir **declara** testes unitários (fase RED). Eles só serão executados após a implementação (fase GREEN); assim, **Executar tudo** não causa falhas propositais. A validação da base real virá em uma segunda suíte independente.

In [ ]:
class TestesUnitariosSDD(unittest.TestCase):
    def test_meta_arredondamento(self):
        self.assertEqual(quantidade_meta(24,.8),20)
        self.assertEqual(quantidade_meta(28,.8),23)
        self.assertEqual(quantidade_meta(5,.8),4)
    def test_meta_argumentos(self):
        with self.assertRaises(ValueError): quantidade_meta(0,.8)
        with self.assertRaises(ValueError): quantidade_meta(10,1.2)
    def test_mcnemar(self):
        self.assertAlmostEqual(mcnemar_exato(4,8),.3876953125,places=9)
        self.assertAlmostEqual(mcnemar_exato(5,3),.7265625,places=9)
        self.assertEqual(mcnemar_exato(0,0),1.)
    def test_pearson_identidade(self):
        p=correlacoes_pareadas(np.array([.1,.2,.3,.4]),np.array([.1,.2,.3,.4]))
        self.assertAlmostEqual(p['pearson_r'],1.,places=9)
    def test_integridade_respostas(self):
        ok=pd.DataFrame([{'ID':'A','Ano':'1º ano','Aplicacao':'contraprova','H':'H01',
                          'Acertos':1,'Itens':2,'Erros':1,'Taxa':.5}])
        validar_respostas(ok)
        ok.loc[0,'Erros']=0
        with self.assertRaises(AssertionError): validar_respostas(ok)
    def test_priorizacao_alvo(self):
        aval=pd.DataFrame([{'ID':'A','Ano':'1º ano','Aplicacao':'contraprova','Adequado':1,'Acertos':5,'Itens':6},
                            {'ID':'B','Ano':'1º ano','Aplicacao':'contraprova','Adequado':0,'Acertos':4,'Itens':6},
                            {'ID':'C','Ano':'1º ano','Aplicacao':'contraprova','Adequado':0,'Acertos':3,'Itens':6}])
        resp=pd.DataFrame([{'ID':'B','Ano':'1º ano','Aplicacao':'contraprova','H':'H01','Acertos':0,'Itens':2,'Erros':2},
                           {'ID':'C','Ano':'1º ano','Aplicacao':'contraprova','H':'H02','Acertos':1,'Itens':2,'Erros':1},
                           {'ID':'A','Ano':'1º ano','Aplicacao':'contraprova','H':'H03','Acertos':0,'Itens':1,'Erros':1}])
        ids=selecionar_grupo_alvo(aval,alvo_n=2,corte=5)
        self.assertEqual(ids,['B'])
        pr=ranking_habilidades(resp,alvo_ids=ids)
        self.assertEqual(pr.iloc[0]['H'],'H01')
    def test_poisson_binomial(self):
        pmf=poisson_binomial([.3,.4])
        np.testing.assert_allclose(pmf,[.42,.46,.12])
        self.assertAlmostEqual(sum(pmf),1.)
        self.assertAlmostEqual(prob_alcancar_meta([1.,.3,.4],2),.58)
    def test_cenario_limites(self):
        for p in (0,.2,.5,1):
            assert 0 <= chance_converter(2,3,p) <= 1
        self.assertEqual(chance_converter(3,2,1),0)
        self.assertEqual(chance_converter(1,2,1),1)
        with self.assertRaises(ValueError): chance_converter(2,2,-.1)

## Etapa 3 — Implementação dos contratos (fase GREEN)

Funções puras, sementes reprodutíveis e validações explícitas. O conjunto de dados embutido é lido posteriormente, sem modificar a planilha-fonte.

In [ ]:
def quantidade_meta(n, fracao=.80):
    if not isinstance(n, (int,np.integer)) or n<=0 or not 0<=fracao<=1:
        raise ValueError("É necessário n>0 e meta entre 0 e 1.")
    return math.ceil(n*fracao-1e-12)

def mcnemar_exato(melhoraram, pioraram):
    b,c=int(melhoraram),int(pioraram)
    if min(b,c)<0: raise ValueError("Contagens inválidas")
    return float(binomtest(min(b,c),n=b+c,p=.5,alternative='two-sided').pvalue) if b+c else 1.

def correlacoes_pareadas(x,y):
    x,y=np.asarray(x,dtype=float),np.asarray(y,dtype=float)
    if len(x)!=len(y) or len(x)<4 or not (np.isfinite(x).all() and np.isfinite(y).all()):
        raise ValueError("Pares finitos e pelo menos 4 observações são necessários")
    if np.std(x)==0 or np.std(y)==0:
        return dict(pearson_r=np.nan,p_pearson=np.nan,spearman_rho=np.nan,p_spearman=np.nan,ic95_inferior=np.nan,ic95_superior=np.nan)
    r,p=pearsonr(x,y);rho,ps=spearmanr(x,y)
    r_safe=np.clip(float(r),-1+1e-12,1-1e-12)
    z=np.arctanh(r_safe);se=1/np.sqrt(len(x)-3)
    return dict(pearson_r=float(r),p_pearson=float(p),spearman_rho=float(rho),p_spearman=float(ps),
                ic95_inferior=float(np.tanh(z-1.96*se)),ic95_superior=float(np.tanh(z+1.96*se)))

def validar_respostas(d):
    assert d[['ID','Ano','Aplicacao','H']].notna().all().all(),"Chaves vazias"
    assert not d.duplicated(['Ano','ID','Aplicacao','H']).any(),"Habilidade duplicada por avaliação"
    assert ((d['Itens']>0)&(d['Acertos']>=0)&(d['Acertos']<=d['Itens'])).all(),"Itens/acertos inválidos"
    assert (d['Erros']==d['Itens']-d['Acertos']).all(),"Erros inconsistentes"
    if 'Taxa' in d.columns:
        assert np.allclose(d['Taxa'],d['Acertos']/d['Itens'],atol=.005),"Taxas inconsistentes"
    return True

def selecionar_grupo_alvo(a,alvo_n,corte):
    if (a['Adequado']==1).sum() >= alvo_n:return []
    d=a.loc[a['Adequado']==0].copy()
    d['deficit']=(corte-d['Acertos']).clip(lower=0)
    d=d.sort_values(['deficit','ID'],kind='stable')
    return d['ID'].head(alvo_n-int((a['Adequado']==1).sum())).tolist()

def ranking_habilidades(respostas,alvo_ids):
    d=respostas.copy()
    grp=d.groupby('H',as_index=False).agg(Acertos=('Acertos','sum'),Itens=('Itens','sum'),Erros=('Erros','sum'))
    focal=d[d['ID'].isin(alvo_ids)].groupby('H')['Erros'].sum()
    grp['Erros_alvo']=grp['H'].map(focal).fillna(0).astype(int)
    grp['Taxa_acerto']=grp['Acertos']/grp['Itens']
    grp['Alvo_n']=len(alvo_ids)
    # Maior número de oportunidades corrigíveis no grupo de menor distância até o corte.
    return grp.sort_values(['Erros_alvo','Taxa_acerto','H'],ascending=[False,True,True],kind='stable').reset_index(drop=True)

def chance_converter(deficit,erros_disponiveis,p):
    if not 0<=p<=1 or deficit<0 or erros_disponiveis<0:raise ValueError('Hipóteses inválidas')
    return 1.0 if deficit<=0 else (float(binom.sf(deficit-1,erros_disponiveis,p)) if erros_disponiveis else 0.)

def poisson_binomial(probs):
    dp=np.array([1.])
    for p in probs:
        if not 0<=p<=1:raise ValueError('Probabilidade fora de [0,1]')
        dp=np.convolve(dp,np.array([1-p,p]))
    return dp

def prob_alcancar_meta(probs,alvo_n):
    dp=poisson_binomial(probs)
    return float(np.sum(dp[alvo_n:])) if alvo_n<len(dp) else 0.

In [ ]:
def executar_testes(classe):
    suite=unittest.defaultTestLoader.loadTestsFromTestCase(classe)
    res=unittest.TextTestRunner(verbosity=2,stream=sys.stdout).run(suite)
    assert res.wasSuccessful(), "SDD/TDD: ao menos um teste falhou. Não interprete os indicadores."
    return res

_ = executar_testes(TestesUnitariosSDD)
print('TDD GREEN: testes unitários concluídos.')

## Etapa 4 — Carregamento verificável de dados

A cópia pseudonimizada incorporada neste notebook permite executá-lo imediatamente. Para reprocessar a planilha original ou atualizada, altere a configuração abaixo. A troca de origem **não** contorna validações: arquivo com esquema diferente falhará explicitamente.

In [ ]:
# SELECIONE UMA FONTE: "embutida" (recomendado para demonstração), "upload" ou "drive".
FONTE = "embutida"
ARQUIVO_DRIVE = "/content/drive/MyDrive/ChatGPT/CAEd_Airam_Veras_2026_Integrado_DADOS_BRUTO.xlsx"
META = 0.80
HABILIDADES_CENARIO = 3  # top-k habil. auditadas no grupo de conversão
PROBABILIDADES_HIPOTETICAS = [0.25,0.50,0.75,1.0]
FONTE_XLSX_SHA256_ORIGINAL = "8ee603b830d355fe26634822e21d558abb83e2efc22221bb82d74f0fc6b6c383"
PACOTE_EMBUTIDO = """eNrVnc1uHEeWhV+lQGCAGYAj5D0RGT/qFUHJFt22JOunPY2GYVSLZU/NkCy5+NOCG36EeQivBuh9b3o5erGpEmVR1SaRxYjMj8qVRLLIeyMqMs65N6LO+evO3h/2vjzY23/y8Pl3Bw927v/przvrf3Zsr2lsZ3dn72Sx/ur//jGZrv63u/PifHk8ff+dvcdP1q94fTR/NX01Xb9uOft+tpydvJpP1z+4mB7Np4fr7z+fH6++8Xh+MTtafbV3OPvxfP2D1WtezZZni9Od+5Z3dw7OZier/0q7Vy+5b7s7n02PzqbHk9eTV4vl2WznfvPzblWWrxYnZ8vp6+XiAkxT4GSq4bLccjIPTs5my+PZ4fztL8v55pT6G5Jtfpus20jWjWJ9OnJ9lr/znpzMwGVZ87Cn0jTbUUxmewcPu9v+YfcbyYZRTGm4gykN20+pbSQbSUhS6ZTGO5jSdvsp1UayaaAp7Qc+a3O9k700DzSlD2bfT0+nP8yON9K07WezrUzz7papNfwydYUsryDXu0MpI0smWeEjZTamtaoxoJRpFFWoufHgk7kRcSnzI9pR0SKqfJdqR7RW2xH1TSzwE6tSQA3oWi3e/slSqphKWxzTKk0j2lHTmCY2j6GTYmOqp9SMobmvZkSESjYekJKN6PEXWU9ZLF2rGgVHlRvFg+9GQaLkR9HpE3sGVfQEaeNcXL9JU8NsoG3JZl+WK/60a+N0/NOfUg01pUMk6waa2P6aZ2VZ4tuoNs7IgclMXJZ3MpntQJN5LSbF7Z+fXJnllpN5bZq3ecw3n/MwiqUZ7mDrLOo8auN4HMMkMFf8kEQbB+PDr9LynSmTO1MZWy5L8y72+Y9OmT/hnakgyzuZTCMnM3BZ3sVuZBoDNzaNYzIduWnm7ffMWJnlnUwmSuGLGyHWjgHOrUUnsxiBwih2o4DWQ9Zs/6CHjTzvgLwXnXGW5Vo1p6WE09J4CqKCXGse+mKmlEeBmhldoLd45m3joVczjs1JKIsvRXfZKB4haQz1pTQKRifyuKDs1LUsS/yDf9o4zvyEO9zyI9ncR1FsqB1FgSmy2Cg+FVYY0Umr4hjqN43joEBpFDiURtFZ0KdM37/d3Xn28PnTJ89f7G0nk3Fzmo9W33l0+Tsf3tqrXHZ3Hi6X7/N7MX2zGrvd65a76IomNJq7Ppr7EM1+jdbcCyFWx/Mb6++aeLqK55yrjtd+HM9dE6/X2QxotIiulIRGy4NF29jVr3nAe33brgkmMpgjg/kB95JrwrXk2MKA+9Y14SK5/MufbNWiqV0T7aNV0lZHExqtBk0Fo6lq0RQYXzmeqhZPm2vWykezWR+NferyYNF6xVOReCoST1WJp0MHa8lggQwWB4OALiy9VTCHVqYOrUxdLZbeao24WiS9JbI5tC51tTgKjC6iKyWh0fJg0XpFUkciqSOR1JFI6kgkdZVI2vOT3YWlt6LIjqxLPYqlHsVSj2Kph7HU12LpLatgj1alHsVSX4ult+KvHsVST2KpJ7HUV2Jpzw9AF5o214RzN4GAJ9HUs2jqyS6vJ9G0Rbu8LdrlbeEubwt3eVu4y9vC1WmL9nlbtDptUURtKxG154XZK6a2ZH3aViLq8BPZkmMb8unutdvbkpga0Ao1oJgaYEwNMKYGuEYNaI0a0Bo1oIgaUEQNLKKGSkQdPpwbbFH2fBspkJgaWEwNZJ0aSEyNaJ0aUUyNMKZGGFMjXKdGFFNjLaYOvlbYpy4PFq3Xvm8ka9RI4mkkz1AjiaaRvI0USSyNJJYmtD5N6AlqQk9QUy2SDh6tpjZNcL83oUiaUCRNKJImEkkTiaSJPUFN5AlqIrE0sZVpIu8jJRJNM4qmGUXTDFemGa5MM4yoGUbUjPZ7M3onKaOImisR9VZAkCsRtedV0oWpw4cb8hnvQtXhRxfYcOW4mkFcvdZueLCOb1E0odEqHrmieBXPXFG8Fh5fgOMVP3ZF0djnLg8Wrd9z1JJwGnCd9FutloTz7OiK69WSYIEdWxwMDvr8LKoZiquoSpIZ2f21ao2k2xQFZmytaqhGkhnZ+zVUI8lQjSQjNZKM1EgyUiPJDDxFNVIhyYxFUlIhyUiFJBPZ+TWhSCoUSWF9JIP1kQzVRzKRXV9D9ZEM1UcyUh/JSH0kI/WRjNRHMlIfyUh9JBOJpCKRFNVHMof2elF9JHNoTepgHHUojjq0JkXVkQxVR7JadaSel0nPXV5SIckcenpqpEaSsRpJRmokmSN7vKhGkqEaSYZqJJknb/iah/EUVUgyVCHJPIqnqEKSefA2knkWTVmNJPMsnpIqSebJ6tSTJ6akRpKhGkmGaiQZrJFksEaSwRpJ1qKIiiokGaqQZKhCkrEKSUYqJBmpkGQt+IkZI/WRrEWdZYzURzJSH8lQfSQLaHUa0Oo0oNVpILXwDVVGMlQZyVBlJEOVkSywvd7AVqeBrU4DW50G9LMyxqojWSARNZD9XlQdyVB1JIPVkQxWRzJYHclQdSSLaIWKqiMZqo5kkcVVUh/JIvoJVIvs52RIjSQjNZKM1EiySCIqqpFkCUXUBCNqghE1wYgK6yQZqpNkiVR1MFQnyUidJEtspZpYTCWVkoxUSrLE9n1JpSQjlZIMVUqyaqWkW01kRvu+mXVuM1gnyTJao6IqSVatkjT4usyDja3nGpXVSTJWJ8lYnSTLJKZmskrNZN+X1EgSqpGkhjxJFayRpIbFVMEaSWpITBWqkCRUIUmoQpIasEYVq4+kBryZJFYdSaQ6khoQT9WAXV81JJ4aWaHKyK6vUG0kGdvzFayOJGN7vkL1kWRkz1eoPpIMrVFlLKIaej9JhtaoMvR+klilJBlYp8rA01ShSklClZKEKiVJ5I1fibzxK1QlSahKkoQiKqqSJLGIKhZRSaUkia1RSa0kiUVTkWhKqiUJVUuSQ9EUVUuSQ9HUoWiKaiXJoWiKaiUJ1UqSI/u9DrzlK4een8qxWEoqJcmR/V5H9nsdiaSoTpJQnSShOknycL/Xw/1eVClJHu32okpJQpWSRColyZNoyuokidVJEqmTJM9Wph684StP9Xm11zRXTWUNjKel0YRGc2g0j0Zr0WgBjRbRaAmNlslol9ZWWDQbbOfaXv9pmGAigw23bW1fcw8T7IYSwAYJFsj3LJLTmMiRZTDY5n418AJBtysTOTJ3PV+160a2GUy1FHJrdlwarXwrFkohVUshBx9bi0YL6ExGdGwJjZbJmaygkKqlkL1G277V1MPGtf0liGFG5shgngxWTiFVSSGHHtlwW1YXhRx66WdwZBUUUiSFFEkhVUkhbxXMoV1Ih3YhHUohHdqFdGgX0qEU0qFdSId2IR1KIR1KIR1KIV0lhRw6WHnp60gK6UgK6UgK6cgupCO7kK6SQg4dLIPBKiikq6SQQ49M5MjKu5AepZAepZAepZAepZAepZAepZAepZAepZAepZAepZAepZCePMj2JIX0JIX0JIX0JIX0JIX0JIX0JIX0JIX0JIX0JIX0JIX0JIVs0YPstpZCDh6tfC9uaynk4GMrP8huaynk4GOLaLSERsvkmtzckocemw23l/RKIdtKCjn0yBw5Mk8GK6eQbSWFHPo9i2SwRAYbbsPqopC9cp8uCjnwNJrIkZUfZIdaCnmrLSSgFDKgFDKgXciAdiFDLYUcfJVEdJUkdGyZfN9uquqHGVtFWR/Ig+xAdiED2YUMJIUMJIUM5F3IQHYhA/lxmkBSyEB2IQPZhQxkFzKQXciIUsiIHmRHlEJGlEJGlEJG9CA7oh+niSiFjCiFjOhBdkQpZCS7kJHsQkaSQkbyIDuSFDKSFDKSFDKSFDKSFDKSXchIdiEj2YWMZBcykbV8Ij/XmMiNOJEbcSI34kRuxInciBN5oyiRG3Eia/lE1vKJrOUTuRFn9FJ6Rmv5jNbyGa3lM1rLZ7SWz+il9IxeSs9oLZ/RWj6jtXwma/lM1vKZpJCZpJCZpJCZvFGUSWmMTNbymaSQmazlM1nLZ7KWzyCFtIakkEXRhEZzaDSPRmvRaAGNFtFoCY2WyWjlFLIo2nB7SZ9dyJJgxTeKSoI5cmSeDNYOBm19fq6xJFgkpzGRwTI4jeVdyJJgRo5M5MjKKaSRN4oM9Xgw1OPBUI8HQz0eDPV4MCNvFJmRN4oM9Xgw1OPBDKWQBgr0moEH2UZ6PBjp8WCkx4MZeJBtpMeDkR4PZmAX0kiPByM9HszALqSRHg+GejyYUAqJejyYUAqJejwY6vFgQruQQruQqMeDoR4Phno8mMguJOnxYKTHg5EeD0Z6PBjp8WAiKaRICimSQpIeD0Z6PBjp8WAiKaRDu5Cox4OhHg+GejwY6vFgqMeDObQL6dAuJOrxYKjHg6EeD+bILiTp8WAOVFczB0pjGOnxYKTHg5EeD+ZAdTUjPR7MgXchzYF3Ic2RXUjS48E82oVEPR6s2uPhVpsI6vFgHu1CelJdzVCPB0M9Hgz1eDDU48FQjwdr0VvcLbpztejO1aI7V4sWvy1a/LZo8duixS8qLW4tunNVS4v3OpO9np+05PlJS56ftOT5CSktbi15ftKSxW9L3uImpcWtJc9PWlBLwlry/ISUFreAFr/V0uK3AhlUWtxQaXELaPEbSHcaQ6XFDZUWN1Ra3ALpTmMBpZCktLgFkkKS0uIWSApJSosbKS1upLS4BZJCktLiFsjzk0BewSGlxY2UFreIUkhUWtwi2oVEpcUtohQyoucnEaWQqLS4VUuLDx2t3J3GIkohSWlxi+QVnEhewSGlxY2UFrdIXsEhpcWNlBY3UlrcSGlxiySFJKXFLZIUMqG3uBNKIRPahUy1FHLwmWzRaAGdyYiOrfxOZEK7kAk9yE7oFZxEUshEUkjSFMESeYubNEWwRFLIBCraWiIpJGmKYImkkKQpgpGmCJZICpnRLmS1KcLg0RwazaPRWjRaQKNFNFpCo2UyWsVBdrUpQq/Rev0gYCYPsmtNEYYemSdHVk4hSVMEI00RLJMfBCRNEYw0RTDSFMEySSEzSCHVkBRSDXkXUg1JIdWQFFIN2YVUQx5kqyEppFBTBDXkQbZQUwShpggydOcysvgVqsUtVItbqBa3UC1uoVrcQrW4hWpxC9Xiln1SO1ef5ycitbhFanGL1OIWqcUtUotbBha/MvAWt0gtbpFa3CK1uEVqcYvU4haqxS1Ui1uoFrdQLW6hWtxCtbgllEIKpZCoFrdQLW6hWtwSSSFJLW6RWtwitbhFanFL4BUckVrcEkkhSS1ukVrcIrW4RWpxi9TilkMpJKrFLVSLW6gWt1AtbqFa3HKkkKIcen6CanEL1eIWqsUtUotbjqSQjqSQjqSQpBa3HNmFJLW45UgKSWpxy5EU0oG3uOVICulAOTKhWtzy6EG2R6/gePQKjidvccuTt7jl0Ss4nrzFLU/e4pYnb3HLk7e4O6P1SiE9+EHAkmCOHJknR9YWd0c8eIu7JFgkpzGRwTIYrIJCerIL6cmDbE92IVtSS0KoKYJatAuJmiIINUUQaoog1BRBqCmCUFMEoaYIatEuJGmKINIUQaQpgkhTBJGmCCJNEUSaIog0RRBpiiDSFEGkKYJIUwSRpghCTREUUAqJmiIINUVQQClkQCkkaoqggHYhUVMEBZRCoqYICuRdSNIUQaQpgkhTBJGmCCJNEUSaIog0RRBpiqBAUshAUkjSFEGBpJDVpgi3mkfUFEERpZCoKYIiSiEjSiFRUwShpgiKKIWMKIWMaBcykl3ISFLISFJI0hRBpCmCSFMEkaYIiiSFJE0RFEkKSZoiKJIUkjRFUEK7kAmVI0NNEYSaIiihFBI1RVBCP06T0I/ToKYIQk0RhJoiKJEUMpEUkjRFUCIpJGmKINIUQaQpghJJIUlTBCWSQpKmCCJNEZTILiRqiiDUFEGoKYJQUwShpghCTRGEmiIINUUQaoog1BRBw5sifLu783Tv2cPn361j/ul9aNtrmishNLsKfflnJ6v9crXv5fd/+/JL6ernl2l9/JJfv/PuVYezH8+nh4v3f+ajb3z4vQ+TsM5EXZmo6c7El2XSbGTiepiTq2Rr5sR3ZhK65yT1kUnbRyauj3cn9JFJ6COT2Lli1Z1J20cmqXNOfC/rpOlcJ7kzE+tlTpquObGmMxPXy4rtzqRzj5Ux68TUx4rtY7c318eKDX28O76HdXL1BlY8O9b2MSe+jznp3mO718nVUqqZk8499mrbGpYVWOpjP+nl3cl9IGAf+4maPjhbH1gs62HF9vLuqHOPtdjLftK5x8r18O5s8xR3Z+L7YAWxdj/RRr2jGzJpgd1eG/VOeSb1TEkb9Y7K653qdaKNekfFu30/mbRdmXQ/xL28N6GHGanHP21UOypnJ7mPJyd3ZmIAO9FGjXHH69W697UAMHtt1Bi6w46SNmqMGzLJzIK1zs1km7Ir9TEloY83p+lhO7Hu7WQLOm09bLHXe2lDpah1WzLebsE2PcyIujeTOBitt26R5dvCX+wjk87NZBsy3fSRia+nJr2skl52tT42enXuakiD7cZbjHcBfko9rNdyzHl33nTw+OGX706bflP6XUyP5qtfWJ9S+avfP73Elg9fTv5l5763e2G1nr+anU3XX6bVzx9PjldfXQ5gf3FyMVueLmbvf3d/sTybTRZ/Pp0tL1Z/4vLh/3VYx/OTyfToYrH+1fVc/WZ6PkorbaaV/imt4O8p35CW20yrvS6rcF1W1q7n7cGzgz88/G7v5YODF0+eHey9m8DPVhM7W+X65Wx+dr6cTmyxTnmyP391tJgcrFI/eDB5sJxfrF9jy/n+mz9/8dnT/PuLv4TTb77+j9OX5+6HN988+vzrn/741X8/Wb382ez0/OjsXSo78pPph4Hv7jy5zPLy4HFv+eP5/GIxmR++/dvJ2fzVKuxkf+/h4XcP1q9++Ob1amDvpuRf3b/d+2n++neT09nx5HRxPF1OThYX0+PZOu/1XP86gq+mZ7Pj6epPTSfLxdn50fSwYzRP/3P5aPbFk3/ff/r16z82bz774fz3Z2H/p9nX3+998Xj60zf/NJrm5sHsr1N4+4/VOp2fHK4T+PLt/zz+/OXe5OmTZy9efv7y4fO93Xf5v1ocnZ9MTyePfrd65cXbX47mq995PV3N++vl4r9mb//37d9np5PD2eTdYN7+sh7Nzs/f/vz/JB3Hlw=="""

def carregar_fontes(fonte=FONTE):
    if fonte == "embutida":
        dados=json.loads(zlib.decompress(base64.b64decode(PACOTE_EMBUTIDO)).decode('utf-8'))
        base={k:pd.DataFrame(v) for k,v in dados.items()}
    else:
        if fonte=="upload":
            try:from google.colab import files
            except ImportError as e:raise RuntimeError('FONTE=upload só funciona no Colab; use FONTE=embutida localmente') from e
            uploads=files.upload()
            if not uploads:raise ValueError('Nenhum Excel enviado')
            nome=next((a for a in uploads if a.lower().endswith('.xlsx')),None)
            if not nome:raise ValueError('Anexe um .xlsx com as abas exigidas')
            caminho=nome
        elif fonte=="drive":
            from google.colab import drive
            drive.mount('/content/drive')
            caminho=ARQUIVO_DRIVE
        else:raise ValueError('Fonte deve ser embutida, upload ou drive')
        base={k:pd.read_excel(caminho,sheet_name=k) for k in
             ['AVALIACOES_ID','RESPOSTAS_ID','PARES_ID','PAINEL','DRIVE_AUDITORIA']}
    for k in base:base[k].columns=base[k].columns.astype(str).str.strip()
    return base

base=carregar_fontes()
a=base['AVALIACOES_ID'];h=base['RESPOSTAS_ID'];pares=base['PARES_ID']
print('Fonte:',FONTE,'| registros de avaliações:',len(a),'| registros de habilidades:',len(h),'| pares:',len(pares))
display(base['DRIVE_AUDITORIA'])

## Etapa 5 — TDD de integração: testes contra os dados reais

Testes para garantir que cada linha da aba `RESPOSTAS_ID` corresponde a uma única combinação **estudante × aplicação × habilidade**, inclusive somando corretamente os itens em `AVALIACOES_ID`. Valores de referência das quatro exportações devem ser reproduzidos, desde que `FONTE="embutida"`. Em fontes novas, os testes estruturais continuam obrigatórios.

In [ ]:
COLUNAS_EXIGIDAS={
  'AVALIACOES_ID':{'ID','Ano','Aplicacao','Acertos','Itens','Adequado','Nivel'},
  'RESPOSTAS_ID':{'ID','Ano','Aplicacao','H','Acertos','Itens','Erros'},
  'PARES_ID':{'ID','Ano','Acertos ref','Itens ref','Acertos contra','Itens contra','Adequado ref','Adequado contra'},
  'PAINEL':{'Ano','Avaliados','Adequados','N meta'},
  'DRIVE_AUDITORIA':{'Fonte','Resultado','Observacao'}}
class TestesIntegracaoDados(unittest.TestCase):
    def test_colunas_obrigatorias(self):
        for tab,cols in COLUNAS_EXIGIDAS.items():
            self.assertTrue(cols.issubset(set(base[tab].columns)), f'{tab}: falta {cols-set(base[tab].columns)}')
    def test_identificadores(self):
        self.assertFalse(a[['ID','Ano','Aplicacao']].isna().any().any())
        self.assertFalse(a.duplicated(['Ano','ID','Aplicacao']).any())
        self.assertFalse(pares.duplicated(['Ano','ID']).any())
    def test_integridade_respostas(self):
        self.assertTrue(validar_respostas(h))
    def test_agregacao_acertos(self):
        g=h.groupby(['Ano','ID','Aplicacao'],as_index=False)[['Acertos','Itens']].sum()
        j=a.merge(g,on=['Ano','ID','Aplicacao'],how='left',validate='one_to_one',suffixes=('_av','_h'),indicator=True)
        self.assertTrue(j['_merge'].eq('both').all())
        self.assertTrue(j['Acertos_av'].eq(j['Acertos_h']).all())
        self.assertTrue(j['Itens_av'].eq(j['Itens_h']).all())
    def test_pares_sao_subconjunto(self):
        left=set(map(tuple,a[['Ano','ID']].itertuples(index=False,name=None)))
        self.assertTrue(set(map(tuple,pares[['Ano','ID']].itertuples(index=False,name=None))).issubset(left))
        for row in pares.itertuples(index=False):
            ano,aluno=row[1],row[0]
            group=a[(a['Ano']==ano)&(a['ID']==aluno)]
            self.assertEqual(set(group['Aplicacao']),{'referencia','contraprova'})
    def test_pares_completos(self):
        exp=a.groupby(['Ano','ID'])['Aplicacao'].nunique()
        self.assertEqual(len(pares),int((exp==2).sum()))
    def test_meta_observada(self):
        for ano,sub in a.query('Aplicacao=="contraprova"').groupby('Ano'):
            linha=base['PAINEL'].loc[base['PAINEL']['Ano']==ano].iloc[0]
            self.assertEqual(len(sub),int(linha['Avaliados']))
            self.assertEqual(int(sub['Adequado'].sum()),int(linha['Adequados']))
            self.assertEqual(quantidade_meta(len(sub),META),int(linha['N meta']))
    def test_casos_zerados(self):
        zero=a.query('Aplicacao=="contraprova" and Acertos==0')
        assert len(zero) >= 0
        self.assertFalse(zero['ID'].isna().any())
    def test_nao_duplicacao_fonte_drive(self):
        # O repositório embutido contém somente registros das 4 aplicações, sem cópias das pastas duplicadas.
        self.assertFalse(a.duplicated(['Ano','ID','Aplicacao']).any())
    def test_tamanho_pares(self):
        if FONTE=='embutida':
            self.assertEqual(len(a),102)
            self.assertEqual(len(h),1054)
            self.assertEqual(len(pares),49)
            self.assertEqual(pares.groupby('Ano').size().to_dict(),{'1º ano':24,'2º ano':25})

_ = executar_testes(TestesIntegracaoDados)
print('AUDITORIA CONCLUÍDA: integridade entre RESPOSTAS_ID e AVALIACOES_ID comprovada.')

## Etapa 6 — Estatística descritiva e meta mínima

`N_alvo = ceil(0,8 × N_avaliados)`. A meta expressa o resultado classificatório **na mesma métrica** da avaliação usada. O quantitativo de acertos adicionais é o **mínimo aritmético** para o grupo de conversão selecionado, não a probabilidade de sucesso nem a promessa de aprendizagem.

In [ ]:
resumo=[];grupos={};cortes={}
for ano,sub in a.query('Aplicacao=="contraprova"').groupby('Ano',sort=True):
    n=len(sub);base_adequados=int(sub['Adequado'].sum());alvo=quantidade_meta(n,META)
    adeq=sub.loc[sub['Adequado']==1,'Acertos'];nao=sub.loc[sub['Adequado']==0,'Acertos']
    # Corte observado é válido nesta base se todos adequados estão acima de todos não adequados.
    corte=int(adeq.min())
    if len(nao) and int(nao.max())>=corte:
        raise ValueError(f'Classificação e acertos se sobrepõem em {ano}; suspender projeções por corte.')
    cortes[ano]=corte
    alvo_ids=selecionar_grupo_alvo(sub,alvo,corte);grupos[ano]=alvo_ids
    foco=sub[sub['ID'].isin(alvo_ids)]
    deficit_min=int((corte-foco['Acertos']).clip(lower=0).sum())
    resumo.append(dict(Ano=ano,Avaliados=n,Adequados=base_adequados,Taxa_adequados=base_adequados/n,
                       Meta=quantidade_meta(n,META),Conversoes_faltantes=max(alvo-base_adequados,0),
                       Corte_empirico=corte,Acertos_minimos_foco=deficit_min))
resumo=pd.DataFrame(resumo)
display(resumo.style.format({'Taxa_adequados':'{:.1%}'}))

for r in resumo.itertuples():
    plt.figure(figsize=(6,3))
    plt.bar([f'Adequados atuais','Meta ({META:.0%})'],[r.Adequados,r.Meta])
    plt.ylim(0,r.Avaliados+2);plt.ylabel('Estudantes')
    plt.title(f'{r.Ano}: {r.Conversoes_faltantes} conversões necessárias')
    for i,v in enumerate([r.Adequados,r.Meta]):plt.text(i,v+.4,str(v),ha='center')
    plt.tight_layout();plt.show()

## Etapa 7 — Contraprova estatística

- **Pearson r:** intensidade da associação linear dos **percentuais** individuais nas duas aplicações, após pareamento pelo ID.
- **Spearman ρ:** associação monotônica, menos sensível a extremos/linearidade.
- **IC95% de Pearson:** transformação z de Fisher (aproximada sob pressupostos).
- **McNemar exato:** compara mudanças de **classificação binária** em pares discordantes.
- **Valor-p:** fornece evidência estatística sob a hipótese nula; **não** prova causalidade, equivalência dos instrumentos nem aprendizado.

In [ ]:
contra=[];pareados_pct={}
for ano,p in pares.groupby('Ano',sort=True):
    x=p['Acertos ref'].to_numpy()/p['Itens ref'].to_numpy()
    y=p['Acertos contra'].to_numpy()/p['Itens contra'].to_numpy()
    c=correlacoes_pareadas(x,y)
    b=int(((p['Adequado ref']==0)&(p['Adequado contra']==1)).sum())
    d=int(((p['Adequado ref']==1)&(p['Adequado contra']==0)).sum())
    contra.append({'Ano':ano,'Pares':len(p),'Melhoraram classe':b,'Pioraram classe':d,'McNemar p':mcnemar_exato(b,d),**c})
    pareados_pct[ano]=(x,y)
    fig,ax=plt.subplots(figsize=(5.5,4));ax.scatter(x*100,y*100,alpha=.75)
    ax.plot([0,100],[0,100],linestyle='--',linewidth=1)
    ax.set(xlim=(0,105),ylim=(0,105),xlabel='Referência (% de acertos)',ylabel='Contraprova (% de acertos)',
           title=f'{ano} | pares={len(p)} | r={c["pearson_r"]:.3f}')
    plt.tight_layout();plt.show()
contra=pd.DataFrame(contra)
display(contra.round(4))
assert abs(contra.loc[contra['Ano']=='1º ano','pearson_r'].iloc[0]-0.1074)<.005
assert abs(contra.loc[contra['Ano']=='2º ano','pearson_r'].iloc[0]-0.5864)<.005

## Etapa 8 — Priorização por habilidade (foco em conversões)

Ordenação transparente: selecionar alunos mais próximos do corte para cobrir o número de conversões faltantes; calcular para cada código H a **quantidade de respostas erradas nesse grupo**; ordenar por essa quantidade, desempatar por menor taxa de acerto na turma. A análise operacional **não considera códigos H como nomes de competências** sem a matriz pedagógica.

> A correlação entre sucesso em H e adequação total pode ser mecanicamente elevada, porque H compõe a própria pontuação total. Ela não deve ser usada como prova de causalidade nem como único ranking.

In [ ]:
rankings={};foco_individual=[]
for ano,sub in a.query('Aplicacao=="contraprova"').groupby('Ano',sort=True):
    hh=h[(h['Ano']==ano)&(h['Aplicacao']=='contraprova')]
    rank=ranking_habilidades(hh,grupos[ano]);rankings[ano]=rank
    display(Markdown(f'### {ano} — prioridade por oportunidades no grupo-alvo'))
    display(rank[['H','Acertos','Itens','Erros','Taxa_acerto','Erros_alvo','Alvo_n']].style.format({'Taxa_acerto':'{:.1%}'}))
    plt.figure(figsize=(8,3.8))
    plt.bar(rank['H'],rank['Erros_alvo'])
    plt.title(f'{ano} | erros no grupo prioritário (n={len(grupos[ano])})')
    plt.xlabel('Código de habilidade CAEd');plt.ylabel('Oportunidades de correção')
    plt.tight_layout();plt.show()
    for id_aluno in grupos[ano]:
        rw=sub[sub.ID==id_aluno].iloc[0]
        erros=hh[(hh['ID']==id_aluno)&(hh['Erros']>0)].sort_values('Erros',ascending=False)
        foco_individual.append(dict(Ano=ano,ID=id_aluno,Acertos=int(rw.Acertos),
                        Faltam_ate_corte=max(cortes[ano]-int(rw.Acertos),0),
                        Habilidades_com_erro='; '.join(f'{s.H}: {int(s.Erros)}' for s in erros.itertuples())))
foco_individual=pd.DataFrame(foco_individual)
display(Markdown('### Fila operacional pseudonimizada — acesso restrito'))
display(foco_individual)

## Etapa 9 — Cenários de projeção, sensibilidade e contraprova

**Hipótese de trabalho:** após intervenção em K habilidades, cada resposta atualmente errada **nessas habilidades** passa a correta independentemente com probabilidade p hipotética. Para cada aluno não adequado com déficit d e e erros disponíveis nessas habilidades, a probabilidade condicional de conversão é `P[Binomial(e,p) ≥ d]`. Para a escola, agregamos as probabilidades individuais via **distribuição Poisson-binomial**, calculada por programação dinâmica exata.

**Isso é análise de sensibilidade, não probabilidade empírica calibrada.** Respostas de um mesmo aluno não são independentes, o impacto da intervenção é desconhecido e *p* não foi estimado em experimento. Valores como 25%, 50% e 75% são **cenários declarados**, não taxas previstas de aprendizagem.

In [ ]:
cenarios=[];limites=[]
for ano,sub in a.query('Aplicacao=="contraprova"').groupby('Ano',sort=True):
    rank=rankings[ano];ks=sorted(set([min(3,len(rank)),min(5,len(rank)),len(rank)]))
    alvo=quantidade_meta(len(sub),META)
    hh=h[(h['Ano']==ano)&(h['Aplicacao']=='contraprova')]
    for k in ks:
        hab=rank['H'].head(k).tolist()
        oportunidades=hh[hh['H'].isin(hab)].groupby('ID')['Erros'].sum().to_dict()
        lim=int(sum(bool(row.Adequado) or oportunidades.get(row.ID,0)>=max(cortes[ano]-int(row.Acertos),0)
                    for row in sub.itertuples()))
        limites.append(dict(Ano=ano,K=k,Habilidades=', '.join(hab),Maximo_teorico=lim,Meta=alvo,
                            Meta_possivel_se_todos_erros_especificos_corrigidos=lim>=alvo))
        for p in PROBABILIDADES_HIPOTETICAS:
            probs=[]
            for row in sub.itertuples():
                if int(row.Adequado)==1:probs.append(1.0)
                else:probs.append(chance_converter(max(cortes[ano]-int(row.Acertos),0),int(oportunidades.get(row.ID,0)),p))
            cenarios.append(dict(Ano=ano,K=k,Habilidades=', '.join(hab),p_hipotetico=p,
                        Adequados_esperados=float(sum(probs)),Pct_esperado=float(np.mean(probs)),
                        Probabilidade_meta_condicional=prob_alcancar_meta(probs,alvo),
                        Maximo_teorico=lim))
cenarios=pd.DataFrame(cenarios);limites=pd.DataFrame(limites)
display(Markdown('### Limites físicos das oportunidades avaliadas'))
display(limites)
display(Markdown('### Sensibilidade probabilística **hipotética**'))
display(cenarios.style.format({'p_hipotetico':'{:.0%}', 'Pct_esperado':'{:.1%}',
                              'Probabilidade_meta_condicional':'{:.1%}', 'Adequados_esperados':'{:.2f}'}))
for ano,dd in cenarios.groupby('Ano',sort=True):
    plt.figure(figsize=(7,4))
    for k,g in dd.groupby('K'):
        plt.plot(g['p_hipotetico'],g['Pct_esperado']*100,marker='o',label=f'{k} habilidades')
    plt.axhline(META*100,linestyle='--',label='Meta 80%')
    plt.xlabel('p: chance hipotética de corrigir cada resposta errada')
    plt.ylabel('% esperado de adequados sob a hipótese')
    plt.title(f'{ano} | sensibilidade — NÃO é previsão validada');plt.legend()
    plt.tight_layout();plt.show()

## Etapa 10 — Auditoria de zeros, fontes e maturidade preditiva

Amostras com prova zerada não podem ser automaticamente interpretadas como dificuldade cognitiva, falta, recusa ou ausência: é necessário conferir a situação do registro com a gestão pedagógica. Ao integrar novos ciclos, o notebook deve exigir **identificador da aplicação, data, instrumento, coorte e protocolo**, além de repetição independente, antes de habilitar ajuste/validação temporal.

In [ ]:
zerados=a[(a['Aplicacao']=='contraprova')&(a['Acertos']==0)][['Ano','ID','Nivel','Acertos','Itens']].copy()
zerados['Alerta']='Auditar ausência, prova em branco, lançamento, condições de aplicação (não inferir causa)'
display(zerados)
quantidade_momentos_reais=2
print('Aplicações disponíveis:',quantidade_momentos_reais,'(sem datas de instrumentos/equacionamento confirmados).')
if quantidade_momentos_reais<3:
    print('BLOQUEIO SDD: não executar previsão temporal ou ML. Somente cenários condicionais.\n'
          'Para validar previsões: ≥3 coletas identificadas no tempo, matriz estável/equacionada, treino/teste temporal e erro fora da amostra.')
display(base['DRIVE_AUDITORIA'])

## Etapa 11 — Exportações reproduzíveis

Gera tabelas agregadas, intervenções por IDs, relatório sintético, especificação dos testes e ZIP. O relatório exportado pode ser analisado pela coordenação. **O arquivo com IDs deve permanecer restrito.**

In [ ]:
OUT=Path('/content/CAEd_Airam_Veras_2026_SDD_TDD') if Path('/content').exists() else Path.cwd()/'CAEd_Airam_Veras_2026_SDD_TDD'
OUT.mkdir(parents=True,exist_ok=True)
exportar={'01_resumo_metas.csv':resumo,'02_contraprova_inferencia.csv':contra,
          '03_cenarios_condicionais.csv':cenarios,'04_limites_oportunidades.csv':limites,
          '05_foco_individual_PSEUDONIMIZADO.csv':foco_individual,
          '06_auditoria_drive.csv':base['DRIVE_AUDITORIA'],
          '07_registros_habilidades_PSEUDONIMIZADO.csv':h,
          '08_registros_avaliacoes_PSEUDONIMIZADO.csv':a,
          '09_pares_validos_PSEUDONIMIZADO.csv':pares}
for ano,rank in rankings.items():
    exportar[f'10_ranking_habilidades_{"1ano" if ano=="1º ano" else "2ano"}.csv']=rank
for nome,tabela in exportar.items():tabela.to_csv(OUT/nome,index=False,encoding='utf-8-sig',sep=';')
(OUT/'ESPECIFICACAO_SDD_TDD.json').write_text(json.dumps({
    'fonte':FONTE,'sha256_excel_original':FONTE_XLSX_SHA256_ORIGINAL,
    'origem':'CAEd Airam Veras 2026; 4 exportações mais auditoria Drive; pseudonimizado',
    'meta':META,'contraprova':'identificação de ciclos não confirmada',
    'modelo_projecao':'poisson-binomial de sensibilidade, p hipotetico; não validado',
    'integridade':'102 avaliações, 1.054 respostas por habilidade, 49 pares na base original',
    'restricoes':['instrumentos não equacionados','corte empírico, não oficial','n pequeno',
                  'dados pseudonimizados requerem acesso controlado','arquivo de Matemática 0 avaliados'],
    'requisitos':list(COLUNAS_EXIGIDAS),'testes':'duas suítes unittest, antes de inferência'},
    ensure_ascii=False,indent=2),encoding='utf-8')

rel=['# RELATÓRIO REPRODUTÍVEL — ESCOLA AIRAM VERAS (2026)',
     'Análise exploratória de Língua Portuguesa; não usar como resultado oficial de Matemática.','']
for rr in resumo.itertuples():
    stats=contra.loc[contra.Ano==rr.Ano].iloc[0]
    rank=rankings[rr.Ano]
    rel += [f'## {rr.Ano}',
            f'- Avaliados: {rr.Avaliados}; adequados: {rr.Adequados} ({rr.Taxa_adequados:.1%}).',
            f'- Alvo: {rr.Meta}/{rr.Avaliados}; faltam {rr.Conversoes_faltantes} conversões; mínimo de {rr.Acertos_minimos_foco} acertos adicionais no grupo-alvo.',
            f'- Corte empírico da contraprova: {rr.Corte_empirico}/22 (não confundir com regra oficial).',
            f'- Pares: {int(stats.Pares)}; r de Pearson={stats.pearson_r:.3f} (p={stats.p_pearson:.4f}); McNemar exato p={stats["McNemar p"]:.4f}.',
            f'- Códigos H para investigar primeiro pelo grupo-alvo: {", ".join(rank.H.head(3))}.',
            '']
rel += ['## Limitações de validade','Correlação não é causalidade. Instrumentos sem equacionamento confirmado.',
        'Projeções são cenários hipotéticos e não modelos calibrados. Confirmar descritores H na matriz oficial.',
        'Checar três respostas zeradas no 2º ano, sem inferir suas causas.','']
(OUT/'RELATORIO_REPRODUZIVEL.md').write_text('\n'.join(rel),encoding='utf-8')
ZIP=OUT.parent/'CAEd_Airam_Veras_2026_Resultados_SDD_TDD.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED,compresslevel=9) as zz:
    for path in OUT.iterdir():zz.write(path,arcname=path.name)
print('Exportação concluída:',ZIP,'\n',*sorted(p.name for p in OUT.iterdir()),sep='\n• ')

## Etapa 12 — Validação do ciclo TDD e uso responsável

**RED → GREEN → REFACTOR:** especificar os testes, implementar, executar os testes e aperfeiçoar os módulos sem alterar os contratos. Para cada novo conjunto do CAEd, incluir um teste que reproduza o defeito detectado antes da correção.

**Rotina pedagógica sugerida (a confirmar com a coordenação):** verificar descritores H e itens oficiais; planejar duas semanas de ensino explícito por habilidade; aplicar miniavaliação independente com itens equivalentes; acompanhar taxa de acerto individual e por grupo; auditar registros em branco; recalcular metas e atualizar o histórico. Evitar extrapolar resultados da disciplina de leitura para Matemática.

**Exercício avançado:** implemente `test_descritores_validados`: ao receber um dicionário H → descritor oficial, recuse códigos desconhecidos. Somente depois apresente nomes de habilidades no plano pedagógico.

In [ ]:
# Use apenas no Google Colab: clique no link de download exibido pelo ambiente.
try:
    import google.colab  # noqa
    from google.colab import files as colab_files
    colab_files.download(str(ZIP))
except ImportError:
    print('Fora do Colab: resultados disponíveis em',ZIP)